<a href="https://colab.research.google.com/github/syedmahmoodiagents/finetuning/blob/main/Pompt_Prefix_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q peft transformers accelerate bitsandbytes --q

In [ ]:
# !pip install -U bitsandbytes

In [ ]:
model_name = "tiiuae/falcon-rw-1b"

In [ ]:
import os, getpass

In [ ]:
os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face token: ")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
from peft import get_peft_model
from peft import TaskType

In [ ]:
# these are meant for either of PromptTuning or PrefixTuning
from peft import PromptTuningConfig, PrefixTuningConfig

In [ ]:
# This is meant for LoRA only
from peft import LoraConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# device_map is on auto mode meaning its automatically set the device mode for which ever is available
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

**for LoRA based Fine Tuning**

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)

In [ ]:
peft_model3.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs"
)

In [ ]:
trainer = Trainer(
    model=peft_model3,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()


**Prompt Tuning**

In [ ]:
config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init="TEXT",
    prompt_tuning_init_text="Summarize the following:",
    num_virtual_tokens=8,
    tokenizer_name_or_path=model_name,
)

In [ ]:
peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters()

**Prefix Tuning**

In [ ]:
config2 = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=8,
    prefix_projection=True,
)

In [ ]:
peft_model2 = get_peft_model(model, config2)
peft_model2.print_trainable_parameters()

**LoRA Tuning**

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)
peft_model3.print_trainable_parameters()

**Implementation of LoRA on Abirate dataset**

In [ ]:
!pip install datasets --q

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

In [ ]:
# Example: load tiny dataset
dataset = load_dataset("Abirate/english_quotes")

In [ ]:
dt = dataset["train"].select(range(100))

In [ ]:
dt.features

In [ ]:
def tokenize(example):
    return tokenizer(example["quote"], truncation=True, padding="max_length", max_length=128)


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
if tokenizer.pad_token is None or tokenizer.pad_token_id == -1:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
dt.map(tokenize)

In [ ]:
tokenized = dt.map(tokenize)

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)

In [ ]:
training_args = TrainingArguments(
    output_dir="./falcon_peft_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-4,
    logging_steps=10,
    fp16=True,
    save_strategy="no",
    report_to=[],
)


In [ ]:
trainer = Trainer(
    model=peft_model3,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is NOT available. Please enable a GPU runtime in Colab.")
    print("Go to Runtime > Change runtime type, then select 'T4 GPU' or another available GPU as the hardware accelerator.")

In [ ]:
peft_model3.save_pretrained("falcon-peft-tuned")

In [ ]:

from peft import PeftModel

model_name = "tiiuae/falcon-rw-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    base_model.resize_token_embeddings(len(tokenizer))
if tokenizer.pad_token is None or tokenizer.pad_token_id == -1:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load the PEFT model from the saved directory
peft_model_path = "./falcon-peft-tuned"
loaded_peft_model = PeftModel.from_pretrained(base_model, peft_model_path)

In [ ]:

# Optional: Merge the LoRA weights into the base model for easier inference
merged_model = loaded_peft_model.merge_and_unload()

print("Model loaded and (optionally) merged successfully!")

Use `merged_model` (or `loaded_peft_model` if didn't merged) for inference, just like with any other Hugging Face model.

In [ ]:
def build_prompt(review: str) -> str:
    return f"""### Instruction:
Classify the sentiment of the review.

### Input:
{review}

### Response:
"""

In [ ]:
import torch

In [ ]:
review = "The product quality was excellent but delivery was late."

prompt = build_prompt(review)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.0
    )

text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
prediction = text.split("Sentiment:")[-1].strip()

print("Prediction:", prediction)